**Setup**

In [29]:
import sys
import os
import shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient

sys.path.insert(0, "/kaggle/input/datasets/noahrushing/llm-gemma-training/llm-training")

secrets = UserSecretsClient()
os.environ["HUGGINGFACE_HUB_TOKEN"] = secrets.get_secret("HF_TOKEN")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Go to Settings > Accelerator > GPU T4 x2")
print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


**Install dependencies**

In [30]:
!pip install -q -r {REPO_ROOT}/requirements.txt

**Copy CSVs into working directory**

In [31]:
raw_dir = Path("/kaggle/working/data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

for path in Path(RAW_CSV_DIR).rglob("*"):
    if path.is_file() and path.suffix.lower() == ".csv":
        shutil.copy(path, raw_dir / path.name)
        print(f"Copied {path.name}")

Copied 7.5in x 8in Hex1 2xBrim Front Right-20250925083521.csv
Copied 7.5in x 8in Hex1 2xBrim Front Right-20250926080727.csv
Copied 7.5in x 8in Hex1 2xBrim Front Right-20251007080848.csv
Copied 7.5in x 8in Hex2 2xBrim Front Left-20250916111958.csv
Copied 017_Sample_033_ros.csv
Copied 7.5in x 8in Hex 2xBrim Rear Right PolyPro-20250924083722.csv
Copied 7.5in x 8in Hex1 2xBrim Front Right-20250915080848.csv
Copied 7.5in x 8in Hex1 2xBrim Front Right-20251002084414.csv
Copied 7.5in x 8in Hex 2xBrim Rear Left PolyPro-20251003080433.csv
Copied 7.5in x 8in Hex1 2xBrim Front Right-20250916080815.csv
Copied 7.5in x 8in Hex 2xBrim Rear Right PolyPro-20251001130529.csv
Copied 7.5in x 8in Hex1 2xBrim Front Right-20250930081421.csv
Copied 7.5in x 8in Hex1 2xBrim Front Right-20250929081954.csv
Copied 7.5in x 8in Hex 2xBrim Rear Left PolyPro-20250924115019.csv
Copied 015_Sample_031_ros.csv
Copied 7.5in x 8in Hex2 2xBrim Front Left-20250915124138.csv
Copied 017_Sample_025_ros.csv
Copied 7.5in x 8in Hex

**Preprocess**

In [43]:
!python {REPO_ROOT}/scripts/preprocess_data.py \
  --input-dir /kaggle/working/data/raw \
  --output-file /kaggle/working/data/processed/operations.jsonl

Wrote 25 records to /kaggle/working/data/processed/operations.jsonl


**Generate synthetic records**

In [44]:
!python {REPO_ROOT}/scripts/generate_synthetic_records.py \
  --input-file /kaggle/working/data/processed/operations.jsonl \
  --output-file /kaggle/working/data/processed/synthetic_operator_records.jsonl

Wrote 21 synthetic operator records to /kaggle/working/data/processed/synthetic_operator_records.jsonl


**Build dataset splits**

In [45]:
!python {REPO_ROOT}/scripts/build_dataset_splits.py \
  --real-file /kaggle/working/data/processed/operations.jsonl \
  --synthetic-file /kaggle/working/data/processed/synthetic_operator_records.jsonl \
  --out-all /kaggle/working/data/processed/dataset_all.jsonl \
  --out-train /kaggle/working/data/processed/dataset_train.jsonl \
  --out-test /kaggle/working/data/processed/dataset_test_manual.jsonl \
  --out-meta /kaggle/working/data/processed/dataset_split_metadata.json

Wrote 46 total examples to /kaggle/working/data/processed/dataset_all.jsonl
Wrote 36 train examples to /kaggle/working/data/processed/dataset_train.jsonl
Wrote 10 test examples to /kaggle/working/data/processed/dataset_test_manual.jsonl
Wrote split metadata to /kaggle/working/data/processed/dataset_split_metadata.json
Held-out groups:
  - 015_Sample_023_ros.csv
  - 015_Sample_031_ros.csv
  - 7.5in x 8in Hex 2xBrim Rear Left PolyPro-20250924115019.csv
  - 7.5in x 8in Hex 2xBrim Rear Left PolyPro-20251003080433.csv
  - 7.5in x 8in Hex 2xBrim Rear Left-20251010095359.csv
  - 7.5in x 8in Hex 2xBrim Rear Right PolyPro-20251001130529.csv


**Prepare fine-tune files**

In [47]:
!python {REPO_ROOT}/scripts/prepare_finetune_files.py \
  --train-input /kaggle/working/data/processed/dataset_train.jsonl \
  --test-input /kaggle/working/data/processed/dataset_test_manual.jsonl \
  --out-train /kaggle/working/data/processed/finetune_train.jsonl \
  --out-test-inputs /kaggle/working/data/processed/finetune_test_inputs.jsonl \
  --out-test-gold /kaggle/working/data/processed/finetune_test_gold.jsonl

Wrote 36 training rows to /kaggle/working/data/processed/finetune_train.jsonl
Wrote 10 test input rows to /kaggle/working/data/processed/finetune_test_inputs.jsonl
Wrote 10 test gold rows to /kaggle/working/data/processed/finetune_test_gold.jsonl


**Prompt baseline: raw_basic**

In [36]:
!python /kaggle/input/datasets/noahrushing/llm-gemma-training/llm-training/scripts/eval_prompt_baselines_gemma.py \
  --mode raw_basic \
  --base-model google/gemma-7b-it \
  --test-inputs /kaggle/working/data/processed/finetune_test_gold.jsonl \
  --output-file /kaggle/working/gemma7b_prompt_raw_basic.jsonl \
  --hf-token $HUGGINGFACE_HUB_TOKEN

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 254/254 [00:05<00:00, 47.10it/s, Materializing param=mo
ex_0001: 13.55s
ex_0002: 22.54s
ex_0003: 20.54s
ex_0015: 14.47s
ex_0016: 17.82s
ex_0021: 22.45s
ex_0026: 22.98s
ex_0027: 24.50s
ex_0039: 23.32s
ex_0043: 24.65s

Wrote predictions to /kaggle/working/gemma7b_prompt_raw_basic.jsonl
Total inference time: 206.83s
Average time per example: 20.68s


**Prompt baseline: basic**

In [37]:
!python /kaggle/input/datasets/noahrushing/llm-gemma-training/llm-training/scripts/eval_prompt_baselines_gemma.py \
  --mode basic \
  --base-model google/gemma-7b-it \
  --test-inputs /kaggle/working/data/processed/finetune_test_gold.jsonl \
  --output-file /kaggle/working/gemma7b_prompt_basic.jsonl \
  --hf-token $HUGGINGFACE_HUB_TOKEN

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 254/254 [00:05<00:00, 47.75it/s, Materializing param=mo
ex_0001: 21.13s
ex_0002: 24.71s
ex_0003: 21.18s
ex_0015: 21.45s
ex_0016: 22.57s
ex_0021: 25.63s
ex_0026: 19.18s
ex_0027: 22.37s
ex_0039: 23.02s
ex_0043: 19.91s

Wrote predictions to /kaggle/working/gemma7b_prompt_basic.jsonl
Total inference time: 221.15s
Average time per example: 22.12s


**Prompt baseline: structured**

In [38]:
!python /kaggle/input/datasets/noahrushing/llm-gemma-training/llm-training/scripts/eval_prompt_baselines_gemma.py \
  --mode structured \
  --base-model google/gemma-7b-it \
  --test-inputs /kaggle/working/data/processed/finetune_test_gold.jsonl \
  --output-file /kaggle/working/gemma7b_prompt_structured.jsonl \
  --hf-token $HUGGINGFACE_HUB_TOKEN

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 254/254 [00:05<00:00, 47.54it/s, Materializing param=mo
ex_0001: 14.13s
ex_0002: 21.09s
ex_0003: 19.33s
ex_0015: 21.60s
ex_0016: 20.25s
ex_0021: 21.52s
ex_0026: 24.15s
ex_0027: 21.19s
ex_0039: 19.11s
ex_0043: 23.84s

Wrote predictions to /kaggle/working/gemma7b_prompt_structured.jsonl
Total inference time: 206.20s
Average time per example: 20.62s


**Prompt baseline: oneshot**

In [39]:
!python /kaggle/input/datasets/noahrushing/llm-gemma-training/llm-training/scripts/eval_prompt_baselines_gemma.py \
  --mode oneshot \
  --base-model google/gemma-7b-it \
  --test-inputs /kaggle/working/data/processed/finetune_test_gold.jsonl \
  --output-file /kaggle/working/gemma7b_prompt_oneshot.jsonl \
  --hf-token $HUGGINGFACE_HUB_TOKEN

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 254/254 [00:05<00:00, 47.74it/s, Materializing param=mo
ex_0001: 15.50s
ex_0002: 38.11s
ex_0003: 23.93s
ex_0015: 16.22s
ex_0016: 25.04s
ex_0021: 38.82s
ex_0026: 26.75s
ex_0027: 25.78s
ex_0039: 25.08s
ex_0043: 32.15s

Wrote predictions to /kaggle/working/gemma7b_prompt_oneshot.jsonl
Total inference time: 267.38s
Average time per example: 26.74s


**Train LoRA**

In [42]:
!python {REPO_ROOT}/scripts/train_gemma_lora.py \
  --config {REPO_ROOT}/configs/gemma_lora.json

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 254/254 [00:05<00:00, 45.43it/s, Materializing param=mo
trainable params: 25,001,984 || all params: 8,562,682,880 || trainable%: 0.2920
{'loss': '3.151', 'grad_norm': '3.007', 'learning_rate': '0.0002', 'epoch': '0.2222'}
{'loss': '2.471', 'grad_norm': '2.479', 'learning_rate': '0.0001867', 'epoch': '0.4444'}
{'loss': '1.951', 'grad_norm': '2.119', 'learning_rate': '0.0001733', 'epoch': '0.6667'}
{'loss': '1.641', 'grad_norm': '2.117', 'learning_rate': '0.00016', 'epoch': '0.8889'}
{'loss': '1.49', 'grad_norm': '2.082', 'learning_rate': '0.0001467', 'epoch': '1'}
 33%|██████████████▎                            | 5/15 [16:18<29:46, 178.62s/it]/usr/local/lib/python3.12/dist-packages/peft/utils/other.py:1394: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-69d83aff-3308eafb56295ade5dd80edf;e6e6aa15-862e-4ee7-87cf-e934bf67d0ab)

Cannot access gated repo for u

**Evaluate fine-tuned model**

In [49]:
!python /kaggle/input/datasets/noahrushing/llm-gemma-training/llm-training/scripts/eval_gemma_lora.py \
  --base-model google/gemma-7b-it \
  --adapter-path /kaggle/working/outputs/gemma7b_lora_manufacturing \
  --test-inputs /kaggle/working/data/processed/finetune_test_inputs.jsonl \
  --output-file /kaggle/working/gemma7b_lora_predictions.jsonl \
  --hf-token $HUGGINGFACE_HUB_TOKEN

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 254/254 [00:05<00:00, 45.24it/s, Materializing param=mo
ex_0001: 52.21s
ex_0002: 52.25s
ex_0003: 52.28s
ex_0015: 52.68s
ex_0016: 53.13s
ex_0021: 53.55s
ex_0026: 53.75s
ex_0027: 36.16s
ex_0039: 53.23s
ex_0043: 52.74s

Wrote predictions to /kaggle/working/gemma7b_lora_predictions.jsonl
Total inference time: 511.99s
Average time per example: 51.20s


**Export readable summaries**

In [50]:
!python /kaggle/input/datasets/noahrushing/llm-gemma-training/llm-training/scripts/export_prediction_summaries.py /kaggle/working/gemma7b_prompt_raw_basic.jsonl
!python /kaggle/input/datasets/noahrushing/llm-gemma-training/llm-training/scripts/export_prediction_summaries.py /kaggle/working/gemma7b_prompt_basic.jsonl
!python /kaggle/input/datasets/noahrushing/llm-gemma-training/llm-training/scripts/export_prediction_summaries.py /kaggle/working/gemma7b_prompt_structured.jsonl
!python /kaggle/input/datasets/noahrushing/llm-gemma-training/llm-training/scripts/export_prediction_summaries.py /kaggle/working/gemma7b_prompt_oneshot.jsonl
!python /kaggle/input/datasets/noahrushing/llm-gemma-training/llm-training/scripts/export_prediction_summaries.py /kaggle/working/gemma7b_lora_predictions.jsonl

Wrote summaries to /kaggle/working/gemma7b_prompt_raw_basic.txt
Wrote summaries to /kaggle/working/gemma7b_prompt_basic.txt
Wrote summaries to /kaggle/working/gemma7b_prompt_structured.txt
Wrote summaries to /kaggle/working/gemma7b_prompt_oneshot.txt
Wrote summaries to /kaggle/working/gemma7b_lora_predictions.txt
